In [ ]:
import optuna
import polars as pl
from zh import (
    CLOSE,
    OPEN,
    OPEN_TIME,
    SHARPE,
    SYMBOL,
    UTC,
    plot,
    scan,
)

data = scan("5m").sort([SYMBOL, OPEN_TIME])
start_date = pl.datetime(2020, 1, 1, time_zone=UTC)
train_date = pl.datetime(2024, 1, 1, time_zone=UTC)
val_date = pl.datetime(2025, 1, 1, time_zone=UTC)
test_date = pl.datetime(2026, 12, 31, time_zone=UTC)

In [ ]:
hold = 2


train_data = (
    data.lazy()
    .sort([SYMBOL, OPEN_TIME])
    .filter(OPEN_TIME >= start_date)
    .with_columns(
        inline_ret_0=(CLOSE / OPEN - 1),
        ret=((CLOSE.shift(1 - hold) / OPEN - 1) / hold).shift(-1).over(SYMBOL),
    )
)


In [ ]:
def get_rets(
    data: pl.LazyFrame | pl.DataFrame,
    long: pl.Expr | None = None,
    short: pl.Expr | None = None,
    start_date: pl.Expr = start_date,
    end_date: pl.Expr = train_date,
    hold: int = hold,
    fees: float = 0.0,
) -> pl.LazyFrame:
    if long is None:
        long = pl.lit(False)
    if short is None:
        short = pl.lit(False)

    rets = (
        data.lazy()
        .filter(OPEN_TIME.is_between(start_date, end_date))
        .select(
            OPEN_TIME,
            SYMBOL,
            pl.when(long & short)
            .then(None)
            .when(long)
            .then(pl.col("ret") - (fees / hold))
            .when(short)
            .then(-pl.col("ret") - (fees / hold))
            .otherwise(None)
            .alias("ret"),
        )
        .group_by(OPEN_TIME)
        .agg((pl.col("ret").mean()).alias("ret"))
        .sort(OPEN_TIME)
    )
    return rets


In [ ]:
def objective(trial: optuna.Trial) -> float:

    inline_ret_0_max = trial.suggest_float("inline_ret_0_max", -0.05, 0.05)
    long = pl.col("inline_ret_0") <= inline_ret_0_max
    rets = get_rets(
        train_data,
        long=long,
        start_date=pl.datetime(2023, 1, 1, time_zone=UTC),
        end_date=val_date,
        fees=10e-4,
    )
    sharpe = (
        rets.group_by_dynamic(OPEN_TIME, every="1d")
        .agg(ret=pl.col("ret").sum())
        .select(pl.col("ret").sum() / pl.col("ret").std())
        .fill_nan(-1000)
    )
    sharpe = sharpe.collect(engine="streaming").item()

    return -sharpe


study = optuna.create_study()
study.optimize(objective, n_trials=100)


In [ ]:
def optimize_one_symbol(symbol: str) -> pl.Expr:
    def objective(trial: optuna.Trial) -> float:
        inline_ret_0_max = trial.suggest_float("inline_ret_0_max", -0.05, 0.05)
        long = pl.col("inline_ret_0") <= inline_ret_0_max
        rets = get_rets(
            train_data.filter(SYMBOL == symbol),
            long=long,
            start_date=pl.datetime(2023, 1, 1, time_zone=UTC),
            end_date=train_date,
            fees=10e-4,
        )
        sharpe = (
            rets.group_by_dynamic(OPEN_TIME, every="1d")
            .agg(ret=pl.col("ret").sum())
            .select(SHARPE)
            .fill_nan(-1000)
        )
        sharpe = sharpe.collect(engine="streaming").item()

        return -sharpe

    study = optuna.create_study()
    study.optimize(objective, n_trials=1000)
    long = pl.col("inline_ret_0") <= study.best_params["inline_ret_0_max"]
    return long


# longs = {}
# for symbol in train_data.select(SYMBOL.unique()).to_series().to_list():
#     longs[symbol] = optimize_one_symbol(symbol)

In [ ]:
def get_rets_for_all_symbols(
    data: pl.LazyFrame | pl.DataFrame,
    longs: dict[str, pl.Expr],
    start_date: pl.Expr = start_date,
    end_date: pl.Expr = test_date,
    fees: float = 0.0,
) -> pl.LazyFrame:
    rets_list: list[pl.LazyFrame] = []
    for symbol, long in longs.items():
        rets = get_rets(
            data.filter(SYMBOL == symbol),
            long=long,
            start_date=start_date,
            end_date=end_date,
            fees=fees,
        )
        rets_list.append(rets)
    all_rets = (
        pl.concat(rets_list)
        .group_by(OPEN_TIME)
        .agg(pl.col("ret").sum())
        .sort(OPEN_TIME)
    )
    return all_rets

In [ ]:
# for symbol, long in longs.items():
#     print(symbol)
#     print(long)

In [ ]:
long:pl.Expr = (
    (pl.col("inline_ret_0") <= study.best_params["inline_ret_0_max"])
)

In [ ]:
plot(
    get_rets(
        train_data,
        # long=long,
        long=(pl.col("inline_ret_0") <= -0.017),
        # short=short,
        start_date=pl.datetime(2023, 1, 1, time_zone=UTC),
        end_date=test_date,
        fees=10e-4,
    )
).properties(width=800, height=400).show()

In [ ]:
train_data.filter(long, OPEN_TIME >= pl.datetime(2023,1,1,time_zone=UTC)).select(OPEN_TIME.unique().count())